# Beyond BLAST Notebook

In [ ]:
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = 0;
using Pkg
Pkg.activate("blast_code"; io=devnull)
Pkg.resolve(; io=devnull)
Pkg.instantiate(; io=devnull)

using Revise 

using Base.Threads, NPZ, DataInterpolations, Interpolations, FastChebInterp
using BenchmarkTools, FFTW, FastTransforms, Dates, TOML, Plots, Plots.Measures
using QuadGK, LaTeXStrings, Tullio, StaticArrays, LoopVectorization, LinearAlgebra
using Unitful, SpecialFunctions, DifferentialEquations, Cosmology, NumericalIntegration
using CSV, DataFrames, JSON, OrderedCollections
;

#### Including the .jl modules

In [ ]:
include("blast_code/src/Blast.jl")
include("blast_code/src/blast_tutorials.jl")
using .Blast
using .blast_tutorials
include("blast_code/src/galaxy_galaxy.jl")
include("blast_code/src/shear_shear.jl")
using .galaxy_galaxy
using .shear_shear
include("blast_code/src/config.jl")
include("blast_code/src/paths.jl")
include("blast_code/src/plot_config.jl")

#### Defining an output folder for each run

In [ ]:
paths = setup_output_directories()

output_dir       = paths.output_dir
plot_subdir      = paths.plot_subdir
Sl_plots         = paths.Sl_plots
Kernel_plots     = paths.Kernel_plots
quantity_subdir  = paths.quantity_subdir
chebcoefs        = paths.chebcoefs
Sl               = paths.Sl

#### Setting run parameters, plot parameters, and grids in k

In [ ]:
grid_data = setup_cosmology_grid() #this sets up the cosmology grid and returns the parameters
plot_theme = setup_plot_theme() #this sets up the plotting theme and returns the parameters
grids = Blast.generate_k_grids(grid_data.kmin, grid_data.kmax, grid_data.Nk, grid_data.Nkp, grid_data.Nkpp; sorting=false) #this returns the k grids for the calculations
params_run = save_run_config(
    output_dir, grid_data.N, grid_data.xmin, grid_data.xmax, grid_data.zmin, grid_data.zmax, grid_data.kmin, grid_data.kmax, grid_data.n_cheb, ℓ, grid_data.Nk, grid_data.Nkp, grid_data.Nkpp,
    grid_data.x, grid_data.z, grids.k_grid, grids.kp_grid, grids.kpp_grid, sorting)
;

In [ ]:
println(grids.k_grid[1])
println(grids.k_grid[end])
println(grid_data.kmin)
println(grid_data.kmax)

$bias = b(z,z^2,z^3)$
comes from [this paper](https://arxiv.org/pdf/1807.10331)

### Galaxy clustering factor

Defining the kernel/window function for a galaxy probe: the window function is \
\
$W(z) = \frac{H(z) n(z) \chi(z)^2 b(z) D(z)}{c} $ \
\
where \
\
$n(z) = A (\frac{z}{z_0})^{\alpha} exp[{-(\frac{z}{z_0})^{\beta}}] $, \
\
with $A = \frac{1.5}{z_0}$, $\alpha = 2$ and $\beta = 1.5$ \
\
$b(z) = b_0 \sqrt{1+z} $ and $b_0 = 1$ \
\
$D(z) = \frac{D(z)^{unnorm}}{D(0)^{unnorm}}$, \
\
with $D(z)^{unnorm} = E(z) \int_z^{\infty} dz' \frac{1+z'}{E(z')^3} $ 

As for the ````gal_prefactor_W_cheb````, it is obtained interpolating each factor, ````bias````, ````growth````, ````nz_norm````, ````chi```` on the ````z````, and then obtaining the total interpolated product ````gal_prefact_W_cheb````.

As for the ````cheb_coeff_gal````, I compute this similarly to Blast: I define a ````plan```` object that takes the ````gal_prefact_W_cheb```` as input, and then returns the ````cheb_coeff```` as the output of the functions ````fast_chebcoefs````

In [ ]:
gal_prefact_W, bias, growth, Hubble_param, nz_norm = galaxy_galaxy.galaxy_prefactor(grid_data.x, grid_data.z, grid_data.cosmo; output_dir=output_dir, plot_style = plot_theme.shared_style);
gal_prefact_W_cheb = galaxy_galaxy.galaxy_prefactor_cheb(grid_data.xmin, grid_data.xmax, grid_data.n_cheb, grid_data.z, grid_data.x, bias, growth, Hubble_param, nz_norm; output_dir=output_dir);

Is it true that

$W(\chi) \approx \sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)$ ?

It seems that the kernel computed on the grid of comoving distances $\chi$, when computed on the Chebyshev nodes, has this behaviour

In [ ]:
println("Kernel on normal nodes at the end of the array: ")
println(gal_prefact_W[end])
println("Kernel on Chebyshev nodes at the start of the array: ")
println(gal_prefact_W_cheb[1])

So one could verify whether the two object are the same just by indexing in a reverse way the Chebyshev object

In [ ]:
gal_prefact_cheb_ordered_like_W = gal_prefact_W_cheb[end:-1:1];

In [ ]:
cheb_coeff_gal = galaxy_galaxy.compute_prefactor_chebcoeffs(gal_prefact_cheb_ordered_like_W; output_dir=output_dir);

In [ ]:
println("Kernel on normal nodes at the end of the array: ")
println(gal_prefact_W[end])
println("Kernel on Chebyshev nodes at the start of the array, now reversed: ")
println(gal_prefact_cheb_ordered_like_W[end])

Here I plot che object on normal nodes vs the N comoving distances $\chi$, and the object on Chebyshev nodes vs the $n_{cheb}$ comoving distances

In [ ]:
x_cheb_plot = Blast.get_clencurt_grid(grid_data.xmax, grid_data.xmin, grid_data.n_cheb)
z_cheb_plot = z_interp.(x_cheb_plot)
plot(x_cheb_plot, gal_prefact_cheb_ordered_like_W, label = L"\sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)", ls=:dash, markersize=0.1, markercolor =:blue)
plot!(grid_data.x, gal_prefact_W, label = L"W(\chi))", lw=1, ls=:dot; plot_theme.shared_style...)

Now I want to compute the relative error between the $W(z)$ and $W(z_cheb)$. To do so, they have to share the same grid. So I bring the normal W(z), previously computed on N nodes, on the Chebyshev grid. 
Note that the object is computed using both $\chi$ and $z$ obtained through get_clencurt_grid, so there's no need to reverse it at the end. 

In [ ]:
galW_computed_on_cheb, _, _, _, _ = galaxy_galaxy.galaxy_prefactor(x_cheb_plot, z_cheb_plot, grid_data.cosmo; output_dir=output_dir);

Now that I have the two object on the same grid, I once again plot the two

In [ ]:
plot(x_cheb_plot, gal_prefact_cheb_ordered_like_W, label = L"\sum_{n=0}^{N_{cheb}-1} c_n T_n(\chi)", ls=:dash, markersize=0.1, markercolor =:blue)
plot!(x_cheb_plot, galW_computed_on_cheb, label = L"W(\chi))", lw=1, ls=:dot)
plot!(dpi = 300, legendposition = :topleft, xlabel = L"\chi [Mpc/h]", ylabel = L"W(\chi) [\mathrm{Mpc}/h]",
        title = "Chebyshev approximation of the galaxy prefactor W"; plot_theme.shared_style...)

In [ ]:
rel_err = (gal_prefact_cheb_ordered_like_W ./ galW_computed_on_cheb) .- 1
rel_err_abs = abs.(rel_err)
rel_err_pct = 100 .* rel_err
rel_err_pct_abs = 100 .* rel_err_abs
;

In [ ]:
plot(
    x_cheb_plot, rel_err_pct,
    label = L"\frac{W_{\mathrm{Cheb}} - W_{\mathrm{true}}}{W_{\mathrm{true}}}\,[\%]",
    lw = 1.5,
    marker = :circle,
    markersize = 2,
    legendposition = :topright,
    xlabel = L"\chi\,[\mathrm{Mpc}/h]",
    ylabel = L"\mathrm{errore\ relativo}\,[\%]",
    title = "Relative error of Chebyshev approximation", size=plot_theme.size_Cl; plot_theme.shared_style...
    )

hline!([0.0], ls = :dash, lw = 1, color = :black, label = false)


In [ ]:
p1 = plot(
    x_cheb_plot, galW_computed_on_cheb,
    label = L"\sum_{n=0}^{N_{\mathrm{cheb}}-1} c_n T_n(\chi)",
    ls = :dash, color = :green)
plot!(p1, x_cheb_plot, gal_prefact_cheb_ordered_like_W, label = L"W(\chi)", lw = 1, ls = :dot, color = :red)
plot!(p1, dpi = 300, legendposition = :topleft, xlabel = L"\chi [Mpc/h]", ylabel = L"W(\chi) [\mathrm{Mpc}/h]", 
        size = plot_theme.size_Cl; plot_theme.shared_style...
        )
    
p2 = plot(
    x_cheb_plot, rel_err_pct,
    label = L"\frac{W_{\mathrm{Cheb}} - W_{\mathrm{true}}}{W_{\mathrm{true}}} [\%]",
    lw = 1.0, marker = :circle, markercolor = :black, markersize = 1, linestyle = :dot, 
    title = "relative error [%]", titlefontsize = 10
)
hline!(p2, [0.0], ls = :solid, lw = 1, color = :red, label = false)

plot(p1, p2, layout = @layout([a; b]), dpi = 300,
     xlabel = L"\chi\,[\mathrm{Mpc}/h]",
    size = plot_theme.size_Cl; plot_theme.shared_style...)

### The dimension of the Chebyshev coefficients is [n_cheb]

In [ ]:
println("size of cheb_coeff_gal: ", size(cheb_coeff_gal))
npzwrite(joinpath(quantity_subdir, "chebcoefs/cheb_coeff_gal.npy"), cheb_coeff_gal)

In [ ]:
n_idx = 0:(grid_data.n_cheb - 1)
plot(n_idx, abs.(cheb_coeff_gal),
     yscale = :log10,
     xlabel = L"{grid_data.n_cheb}",
     ylabel = L"|c_{grid_data.n_cheb}|",
     label = L"|c_{grid_data.n_cheb}| \ \mathrm{di} \ W(\chi)",
     title = "Chebyshev coefficients decay",
     marker = :circle, markersize = 3; plot_theme.shared_style...
     )
savefig(joinpath(plot_subdir, "kernels", "chebcoeff_decay_galaxy.png"))


In [ ]:
function cheb_eval_partial(coeffs, x, xmin, xmax, ntrunc)
    t = @. (2*x - (xmax + xmin)) / (xmax - xmin)
    T0 = ones(length(x))
    if ntrunc == 0
        return coeffs[1] .* T0
    end
    T1 = t
    S = coeffs[1] .* T0 .+ coeffs[2] .* T1
    for n in 2:ntrunc
        T2 = @. 2*t*T1 - T0
        S .+= coeffs[n+1] .* T2
        T0, T1 = T1, T2
    end
    return S
end

chi_cheb_nodes = Blast.get_clencurt_grid_z(grid_data.xmin, grid_data.xmax, grid_data.n_cheb)
x_ref = chi_cheb_nodes
Wref = gal_prefact_W_cheb

truncs = 1:1:length(cheb_coeff_gal)-1
#truncs = [5, 10, 20, 40, 80, 120, 160, length(cheb_coeff_gal)-1]
errs = Float64[]
Wn = zeros(length(x_ref))
for ntrunc in truncs
    Wn = cheb_eval_partial(cheb_coeff_gal, x_ref, grid_data.xmin, grid_data.xmax, ntrunc)
    err = maximum(abs.(Wn .- Wref)) / maximum(abs.(Wref))
    push!(errs, err)
end

truncs_plot = truncs[1:end]
errs_plot = errs[1:end]
yticks_vals = 10.0 .^ (floor(log10(minimum(errs_plot))):ceil(log10(maximum(errs_plot))))

plot(truncs_plot, errs_plot,
     yscale = :log10,
     marker = :circle,
     xlabel = L"N_{trunc}",
     ylabel = L"\mathrm{err}(N_{trunc}) = \frac{\max_i |W_{N_{trunc}} - W(\chi_i)|}{\max_i |W(\chi_i)|}",
     title  = L"\mathrm{err}(N_{trunc}) = \frac{\max_i |W_{N_{trunc}} - W(\chi_i)|}{\max_i |W(\chi_i)|}",
     legend = false,
     framestyle = :box,
     yticks = yticks_vals, size=plot_theme.size_Cl; plot_theme.shared_style...
     )
savefig(joinpath(plot_subdir, "kernels", "chebcoeff_truncation_error_galaxy.png"))

$W_{tilde} = \int dz W(z) j_l(k\chi(z)) j_l(k_1\chi(z))$

$\tilde W_{\ell}^g(k1,k) \approx \sum_{n=0}^{N_{cheb}-1} c_n \int_{z_{min}}^{z_{max}} dz T_n(\hat z) k_1 j_l(k\chi(z)) j_l(k_1\chi(z))$

### The dimension of the $\tilde W$ matrix is [$N_k$, $N_{kp}$, $n_{cheb}$, $\ell$]

In [ ]:
# # N = 2^15+1, Nk = Nkp = Nkpp = 150, N_cheb = 200, length(ℓ) = 100
# W_tilde = zeros(grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, length(grid_data.ℓ))
# W_tilde = npzread("/Users/anvi/Desktop/grid_data.cosmo/notebooks/out/W_tilde_new_funcs.npy")
# ;

In [ ]:
W_tilde = zeros(grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, length(grid_data.ℓ))
elapsed_time = zeros(length(grid_data.ℓ))
println("Dimensions of W_tilde: ", size(W_tilde))
for i in eachindex(grid_data.ℓ)
    # i want the elapsed time for each l 
    t_0 = time()
    W_tilde[:, :, :, i] .= Blast.W_tilde_computation(grid_data.ℓ[i], grid_data.zmin, grid_data.zmax, grid_data.kmin, grid_data.kmax,
                                                 grid_data.Nk, grid_data.Nkp, grid_data.n_cheb, grid_data.N, grids.k_grid, grids.kp_grid, grids.kpp_grid)
    t_end = time()
    elapsed_time[i] = t_end - t_0
    println("Finished computing W_tilde for ℓ = $(grid_data.ℓ[i]). Time elapsed $(round(elapsed_time[i], digits=2))s")
end
;

In [ ]:
npzwrite(joinpath(output_dir, "W_tilde.npy"), W_tilde)

In [ ]:
println("Size of W_tilde: ", size(W_tilde))
#Size of W_tilde: (Nk, Nkp, Ncheb, Nl)

In [ ]:
# println("min = ", minimum(W_tilde), ", max = ", maximum(W_tilde))
# println("mean = ", mean(W_tilde), ", std = ", std(W_tilde))
# if  any(isnan, W_tilde)
#     println("There are NaN values in W_tilde")
# end
# if any(isinf, W_tilde)
#     println("There are infinite values in W_tilde")
# end
# if any(iszero, W_tilde)
#     println("There are zero values in W_tilde")
# end

$\tilde W(k,k_1)$ (like $\tilde W(k,k_2)$) represents the term: \
$\tilde W_{i,p,l}^{(\ell)} = \sum_{m=1}^{N_k} w_{k_m} T_{\ell}(k_m) j_{\ell}(\chi_i k_m) j_{\ell}(\chi_p k_m) $ \
it describes how much two shells at comoving distance $\chi_i$ and $\chi_p$ are correlated to the multipole $\ell$, weighted by the Chebyshev polynomial $T_{\ell}$ on the mode $k$.

### The dimension of $W_{final}^{gal}$ is [$\ell$, $N_k$, $N_{kp}$]

In [ ]:
@tullio W_final_gal[il, ik, ikp] := W_tilde[ik, ikp, ic, il] * cheb_coeff_gal[ic];

In [ ]:
println("Size of W_final_gal: ", size(W_final_gal))
# [ell, k_grid, kp_grid]

In [ ]:
# println("min = ", minimum(W_final_gal), ", max = ", maximum(W_final_gal))
# println("mean = ", mean(W_final_gal), ", std = ", std(W_final_gal))
# if  any(isnan, W_final_gal)   
#     println("There are NaN values in W_final_gal")
# end
# if any(isinf, W_final_gal)   
#     println("There are infinite values in W_final_gal")
# end
# if any(iszero, W_final_gal)
#     println("There are zero values in W_final_gal")
# end

### $k_{grid}$

In [ ]:
idx = sortperm(grids.k_grid)
idx_p = sortperm(grids.kp_grid)

# genera tick automatici alle potenze di 10 nel range dei dati
xticks_vals = 10.0 .^ (floor(log10(minimum(grids.k_grid))):ceil(log10(maximum(grids.k_grid))))
yticks_vals = 10.0 .^ (floor(log10(minimum(grids.kp_grid))):ceil(log10(maximum(grids.kp_grid))))

heatmap(grids.k_grid[idx], grids.kp_grid[idx_p],
        W_final_gal[1,idx,idx_p]/maximum(W_final_gal[1,idx,idx_p]),
        title=L"W_{final}^{gg}"*"(at fixed "*L"ℓ)",
        xscale = :log10, yscale = :log10,
        xlabel=L"k", ylabel=L"k_p",
        size = size_heatmap,
        c = c, 
        xticks = xticks_vals, yticks = yticks_vals; plot_theme.shared_style...
        )


In [ ]:
savefig(joinpath(plot_subdir, "Sl_plots/k_grid_sorted_k_grid_sorted.png"))

In [ ]:
println(size(W_tilde))

In [ ]:
heatmap(1:grid_data.Nk, 1:grid_data.Nkp, W_final_gal[10,:,:]/maximum(W_final_gal[10,:,:]), 
    title = L"W_{final}^{gg}"*"(at fixed "*L"ℓ)", 
    xlabel = L"N_k", ylabel = L"N_{kp}", 
    colorbar = true, c = c, size = size_heatmap; plot_theme.shared_style...
    )


In [ ]:
savefig(joinpath(plot_subdir, "Sl_plots/Nk_Nkp.png"))

In [ ]:
idx_p = sortperm(grids.kp_grid)
yticks_vals = 10.0 .^ (floor(log10(minimum(grids.k_grid))):ceil(log10(maximum(grids.k_grid))))
heatmap(1:grid_data.Nk, 
        grids.kp_grid[idx_p], 
        W_final_gal[90, :, idx_p] / maximum(W_final_gal[90, :, idx_p]), 
        title = L"W_{final}^{gg}"*L"("*"at fixed "*L"ℓ = 90)", 
        xlabel = L"N_k", ylabel = L"\log_{10}(k_{grid})",
        yscale = :log10, colorbar = true, c = c, size = size_heatmap; plot_theme.shared_style...
        )


In [ ]:
savefig(joinpath(plot_subdir, "Sl_plots/Nk_kp_grid_sorted.png"))

In [ ]:
#3D matter power spectrum
pk_dict = npzread("blast_code/data/pk.npz")
Pklin = pk_dict["pk_lin"]
Pknonlin = pk_dict["pk_nl"]
k_pk = pk_dict["k"]
z_pk = pk_dict["z"]
#Interpolating the power spectrum: Linear P(k) - Non-linear P(k)
y_pk = LinRange(log10(first(k_pk)),log10(last(k_pk)), length(k_pk))
x_pk = LinRange(first(z_pk), last(z_pk), length(z_pk))
InterpPmm = Interpolations.interpolate(log10.(Pklin),BSpline(Cubic(Line(OnGrid()))))
InterpPmm = scale(InterpPmm, (x_pk, y_pk))
InterpPmm = Interpolations.extrapolate(InterpPmm, Line())
InterpPmm_nl = Interpolations.interpolate(log10.(Pknonlin),BSpline(Cubic(Line(OnGrid()))))
InterpPmm_nl = scale(InterpPmm_nl, x_pk, y_pk)
InterpPmm_nl = Interpolations.extrapolate(InterpPmm_nl, Line())
power_spectrum(k_pk, χ1, χ2) = @. sqrt(10^InterpPmm(grid_data.z_of_χ(χ1),log10(k_pk)) * 10^InterpPmm(grid_data.z_of_χ(χ2),log10(k_pk)))
power_spectrum_nl(k_pk, χ1, χ2) = @. sqrt(10^InterpPmm_nl(grid_data.z_of_χ(χ1),log10(k_pk)) * 10^InterpPmm_nl(grid_data.z_of_χ(χ2),log10(k_pk)))

In [ ]:
plot(k_pk, power_spectrum.(k_pk, 1000.0, 1000.0), 
     label="Linear P(k)", 
     xscale=:log10, 
     yscale=:log10, 
     title=L"$P(k)$ at $z=0$", titlefontsize=20,
     xlabel=L"$k \; (h/\mathrm{Mpc})$", 
     ylabel=L"$P(k) \; ((\mathrm{Mpc}/h)^3)$", 
     labelfontsize=15)
plot!(k_pk, power_spectrum_nl.(k_pk, 1000.0, 1000.0), 
      label="Non-linear P(k)", 
      xscale=:log10, 
      yscale=:log10, size=plot_theme.size_Cl; plot_theme.shared_style...
      )

get_clencurt_grid produces the node of Clenshaw-Curtis mapped on [$k_{min}$, $k_{max}$]. \
get_clencurt_weights produces the corresponding quadrature weights scaled to the interval [-1,1]. \

In [ ]:
Pk_grid = power_spectrum.(grids.k_grid, grid_data.kmin, grid_data.kmax)
w_k = Blast.get_clencurt_weights(grid_data.kmin, grid_data.kmax, grid_data.Nk)
weight_gal = w_k .* grids.k_grid.^2 .* Pk_grid 
;

###alternative: I'd need to change also the integration of W_tilde
# log_k_grid = range(log(kmin), log(kmax), length=grid_data.Nk)
# k_grid_2 = exp.(log_k_grid)
# dlnk = step(log_k_grid)
# w_log = ones(grid_data.Nk) * dlnk
# w_log[1] /= 2.0
# w_log[end] /= 2.0
# weight_gal = w_log .* k_grid_2.^3 .* Pk_grid

In [ ]:
abstract type AbstractProbe end
struct Galaxy <: AbstractProbe end
struct Shear <: AbstractProbe end
factorial_frac(ℓ) = (ℓ + 2.0) * (ℓ + 1.0) * ℓ * (ℓ - 1.0)
get_ell_prefactor(::Galaxy, ::Galaxy, ℓ) = @. (2 / π) * ones(length(ℓ))
get_ell_prefactor(::Galaxy, ::Shear,  ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Galaxy, ℓ) = @. (2 / π) * sqrt(factorial_frac(ℓ))
get_ell_prefactor(::Shear,  ::Shear,  ℓ) = @. (2 / π) * factorial_frac(ℓ)
pref_gg = get_ell_prefactor(Galaxy(), Galaxy(), grid_data.ℓ)
pref_gg = reduce(vcat, pref_gg)
pref_gs = get_ell_prefactor(Galaxy(), Shear(), grid_data.ℓ)
pref_gs = reduce(vcat, pref_gs)
pref_gg = reduce(vcat, pref_gg)
pref_ss = get_ell_prefactor(Shear(), Shear(), grid_data.ℓ)
pref_ss = reduce(vcat, pref_ss);

In [ ]:
println("SIZES")
println("weight_gal -> ", size(weight_gal))
println("W_final_gal -> ", size(W_final_gal))
println("pref_gg -> ", size(pref_gg))

In [ ]:
S_lkk_gg = zeros(Float64, size(W_final_gal, 3), size(W_final_gal, 3), length(grid_data.ℓ))
@tullio S_lkk_gg[kp, kpp, li] = pref_gg[li] * weight_gal[k] * W_final_gal[li, k, kp] * W_final_gal[li, k, kpp]
npzwrite(joinpath(quantity_subdir, "Sl/S_lkk_gg.npy"), S_lkk_gg)
println("Size of S_l (kp, kpp) (gal-gal): \n(grid_data.Nk, grid_data.Nkp, NL) -> ", size(S_lkk_gg))
;

wavenumber k as a function of $\ell$ through $k \approx \frac{\ell+0.5}{\chi}$ ?

In [ ]:
npzwrite(joinpath(quantity_subdir, "Sl/S_lkk_gg.npy"), S_lkk_gg)

In [ ]:
xref = (grid_data.xmax - grid_data.xmin ) * 0.5
ℓ_to_k = ℓ -> ℓ ./ xref
ℓ_ticks = 1:50:200
k_ticks = ℓ_to_k.(ℓ_ticks)
k_ticklabels = [string(round(k, digits=3)) for k in k_ticks];

In [ ]:
plot(grid_data.ℓ, S_lkk_gg[5,5,:],
      color = colors[1],
      label = L"i, j=5")

plot!(xaxis = L"\ell",
      ylabel = L"S_\ell",
      xscale = :log10,
     )
plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xscale = :log10,
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      label = ""
     )
plot!(label = "Beyond BLAST", 
      size=plot_theme.size_Cl,
      title = L"S_\ell^{gg} (i = j)",
      titlefontsize = 20,
      titleposition = :left ; plot_theme.shared_style...)

In [ ]:
plot(k_grid, S_lkk_gg[:,1,1],
      color = colors[1],
      label = L"i, j=1")

plot!(xaxis = L"k_{grid} \; (h/\mathrm{Mpc})",
      ylabel = L"S_\ell",
      xscale = :log10,
     )
plot!(label = "Beyond BLAST", 
      title = L"S_\ell^{gg} (k, kp = 1, i_{\ell} = 1)",
      titlefontsize = 20,
      titleposition = :left,
      legendposition = :topleft, size=plot_theme.size_Cl; plot_theme.shared_style...)

In [ ]:
plt = plot()

for i in 1:grid_data.Nkp
    plot!(plt, grid_data.ℓ, S_lkk_gg[i,i,:], line_z = i,
          label = L"i = %$(i)",
          color = colors, linewidth = 2)
end

plot!(plt, label="Beyond BLAST", 
     xaxis=L"\ell", ylabel=L"S_\ell", 
     xscale = :log10
    )
plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      xscale = :log10,
      label = ""
     )
plot!(legend = false, colorbar = true, colorbar_title = L"i = j",
      clims = (1, grid_data.Nkp),
      titleposition = :left,
      title=L"S_\ell^{gg} (i = j)", size=plot_theme.size_Cl; plot_theme.shared_style...
     )
plt

In [ ]:
plt = plot()

for i in 1:grid_data.Nk
    plot!(plt, grid_data.ℓ, S_lkk_gg[i,i,:]/maximum(S_lkk_gg[i,i,:]), line_z = i,
          label = L"i = %$(i)",
          color = colors, linewidth = 1)
end

plot!(plt, 
    clims = (1, grid_data.Nk),
    xaxis=L"\ell", yaxis=L"S_\ell", 
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      label = ""
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j",
    title=L"S_\ell^{gg} (i = j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left; plot_theme.shared_style...
    )

plt

In [ ]:
savefig(joinpath(plot_subdir, "Sl_plots/S_lkk_gg_fixed_l_various.png"))

In [ ]:
plt = plot()

a = 1:5:grid_data.Nk
for i in a
    plot!(plt, grid_data.ℓ, S_lkk_gg[i,i,:]/maximum(S_lkk_gg[i,i,:]), line_z = i,
          label = L"i = %$(i)", color = colors, linewidth = 1
          )
end

plot!(plt, 
    xaxis=L"\ell", yaxis=L"S_\ell", 
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))     
      )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"i_k", clims = (1, grid_data.Nkp),
    title=L"S_\ell^{gg} (i = j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left; plot_theme.shared_style...
    )

plt

In [ ]:
i_fixed = 1
for j in 1:grid_data.Nkp
    plot!(plt, grid_data.ℓ, S_lkk_gg[i_fixed, j, :],
          line_z = j, color = colors, linewidth = 2
        )                  
end

plot!(plt,
    xaxis = L"\ell", yaxis = L"S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"S_\ell^{gg} (i = 1,\ j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt


In [ ]:
plt = plot()
jj = 96
i_fixed = 1
for j in 1:jj
    plot!(plt, grid_data.ℓ, S_lkk_gg[i_fixed, j, :] / maximum(S_lkk_gg[i_fixed, j, :]),
          line_z = j, color = colors, linewidth = 2
          )
end

plot!(plt,
    xaxis = L"\ell", yaxis = L"S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"S_\ell^{gg} (i = 1,\ j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
plt = plot()
jj = 100
i_fixed = 50
for j in 1:jj
    plot!(plt, grid_data.ℓ, S_lkk_gg[i_fixed, j, :],
          line_z = j, color = colors,
          linewidth = 2
          )
end

plot!(plt,
    xaxis = L"\ell", yaxis = L"S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
    )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, Nkp),
    title = L"S_\ell^{gg} (i = 10,\ j)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
plot(
    grid_data.ℓ,
    S_lkk_gg[1, 1, :] .* grid_data.ℓ .* (grid_data.ℓ .+ 1),
    xaxis = L"\ell",
    yaxis = L"\ell(\ell+1)S_\ell",
)

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ)),
      label = ""
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, Nkp),
    title = L"\ell(\ell+1)S_\ell^{gg}(k_1,k_2)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

In [ ]:
plt = plot()

for j in 1:Nkp
    plot!(plt, grid_data.ℓ, S_lkk_gg[j, j, :] .* grid_data.ℓ .* (grid_data.ℓ .+ 1),
          line_z = j, color = colors, linewidth = 2
          )
end

plot!(plt,
    xaxis = L"\ell", 
    yaxis = L"\ell(\ell+1)S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, Nkp),
    title = L"\ell(\ell+1)S_\ell^{gg}(k_1,k_2)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
plt = plot()

for j in a
    plot!(plt, grid_data.ℓ, S_lkk_gg[j, j, :] .* grid_data.ℓ .* (grid_data.ℓ .+ 1),
          line_z = j, color = colors, linewidth = 2
          )                  
end

plot!(plt,
    xaxis = L"\ell", 
    yaxis = L"\ell(\ell+1)S_\ell",
    xscale = :log10
    )

plot!(twiny(),
      xaxis = L"k \; (h/\mathrm{Mpc})",
      xticks = (ℓ_ticks, k_ticklabels),
      xlims = (minimum(grid_data.ℓ), maximum(grid_data.ℓ))
     )

plot!(label = "Beyond BLAST", 
    legend = false, colorbar = true, colorbar_title = L"j", clims = (1, grid_data.Nkp),
    title = L"\ell(\ell+1)S_\ell^{gg}(k_1,k_2)",
    size=plot_theme.size_Cl, titlefontsize=20, 
    titleposition = :left;
    plot_theme.shared_style...
    )

plt

In [ ]:
println(size(S_lkk_gg))

In [ ]:
diagS = [diag(S_lkk_gg[:, :, i]) for i in 1:size(S_lkk_gg, 3)]

nshow = 100
idx = round.(Int, range(1, size(S_lkk_gg, 3), length=nshow))

p = plot(
    xlabel = L"k \; (h/\mathrm{Mpc})",
    ylabel = L"S_\ell(k,k)",
    title  = L"S_\ell^{gg}(k,k)\ \mathrm{for\ different}\ \ell\ \mathrm{values}",
    colorbar_title = L"j",
    clims = (1, grid_data.Nkp),                   
    legend = false,
    colorbar = true,
    lw = 2,
    size=plot_theme.size_Cl; plot_theme.shared_style...
)

for ii in idx
    plot!(p, grids.kp_grid, diagS[ii]/maximum(diagS[ii]), line_z = ii, color = colors;
        label = L"\ell = %$(grid_data.ℓ[ii])")
end

p

---